In [1]:
%%capture --no-stderr
%pip install --quiet -U langchain_openai langchain_core langchain_community tavily-python

In [71]:
import os, getpass

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("OPENAI_API_KEY")

In [72]:
api_key = os.getenv("OPENAI_API_KEY")

#setting up chatmodels
from langchain_openai import ChatOpenAI
gpt4o_chat = ChatOpenAI(model="gpt-4o", temperature=0, openai_api_key=api_key)
gpt35_chat = ChatOpenAI(model="gpt-3.5-turbo-0125", temperature=0)

First we need to know about 'State' 
- State is python object that represents shared memory or context across all nodes
- state store 1. input and output of nodes 2. intermediate values 3. Conversational memory 4. External tool usage history


Below is how to use STATE:

In [73]:
from typing import TypedDict, List, Union
from langgraph.graph import StateGraph,START, END

class Istate(TypedDict):
    message:str
    interests:list[str]
    thoughts:list[str]

In [74]:
#NOTE: These functions are called as node which will be used by LLM to process information
def random_node(state: Istate) -> Istate:
    last_message = state['message'][-1]
    n_interests = state['interests'][0]
    new_thought = f"Thinking about: {last_message}"
    return {
        #.get(keyname,value)
        "thoughts": state.get("thoughts") + [new_thought],
        "interests" : state.get("interests") + [n_interests]
    }

In [76]:
graph = StateGraph(Istate)
graph.add_node("think", random_node)
graph.add_edge(START, "think")
graph.add_edge("think", END)
app = graph.compile()
final_state = app.invoke({"message": ["Hello!"], "thoughts": ["I am not cool"], "interests": [""]})
final_state

{'message': ['Hello!'],
 'interests': ['', ''],
 'thoughts': ['I am not cool', 'Thinking about: Hello!']}

final_state = app.invoke({"message": ["Hello!"], "thoughts": ["I am not cool"], "interests": [""]})
final_state

TypedDict:
-TypedDict is a class from Python’s typing module that allows you to define a dictionary with fixed keys and specific value types, similar to structs or interfaces in other languages.

-It helps Python understand:

What keys should exist in a dictionary.

What data types those keys should map to.

In [51]:
#Example of typeDict
from typing import TypedDict

class User(TypedDict):
    name: str
    age: int

def greet(user: User) -> None:
    print(f"Hello {user['name']}, you are {user['age']} years old.")

In [53]:
greet({"name":"rishi", "age":23})

Hello rishi, you are 23 years old.


In [48]:
from typing import TypedDict

class Book(TypedDict):
    title: str
    author: str
    year: int

In [49]:
def print_book(book: Book) -> None:
    print(f"{book['title']} by {book['author']} ({book['year']})")

print_book({"title": "1984", "author": "Orwell", "year": 1949})

1984 by Orwell (1949)


What is Annotated?
-It allows you to attach metadata to a type hint.
-The metadata doesn't affect how Python runs the code — but it can be used by tools or frameworks (like LangGraph) to attach behaviors or validations.

What is add_messages in LangGraph?
-add_messages is a special modifier function provided by LangGraph:
It tells LangGraph:
"This field is a list of messages, and when a node returns messages, they should be appended to this list."

Without add_messages:
You would have to manually do:
def node(state):
    state["messages"].append(AIMessage(content="Hello"))
    return state
merge and append


In [67]:
#reducer example

from typing import Annotated, TypedDict
from langchain.schema import BaseMessage, AIMessage, HumanMessage
from langgraph.graph import StateGraph, END

def keep_last_2(existing: list[BaseMessage], new: list[BaseMessage]) -> list[BaseMessage]:
    return (existing + new)[-2:]

class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], keep_last_2]

#BaseMessage is aseMessage is an abstract base 
#class from LangChain, representing a single chat message in a conversation between a human and an AI.
#Example
# from langchain.schema import BaseMessage, AIMessage, HumanMessage

# chat_history: list[BaseMessage] = []

# chat_history.append(HumanMessage(content="Hello!"))
# chat_history.append(AIMessage(content="Hi there! How can I help?"))

In [68]:
def greet(state: ChatState):
    return {"messages": [AIMessage(content="Hello!")]}

def ask(state: ChatState):
    return {"messages": [HumanMessage(content="What’s up?")]}

def reply(state: ChatState):
    return {"messages": [AIMessage(content="All good here!")]}

def user_follows_up(state: ChatState):
    return {"messages": [HumanMessage(content="Cool, thanks!")]}

In [69]:
graph = StateGraph(ChatState)

graph.add_node("greet", greet)
graph.add_node("ask", ask)
graph.add_node("reply", reply)
graph.add_node("followup", user_follows_up)

graph.set_entry_point("greet")
graph.add_edge("greet", "ask")
graph.add_edge("ask", "reply")
graph.add_edge("reply", "followup")
graph.add_edge("followup", END)

chat_app = graph.compile()

In [70]:
state = {"messages": []}
final_state = chat_app.invoke(state)

print("\nFinal State (should only show last 2 messages):\n")
for msg in final_state["messages"]:
    print(f"{msg.type.upper()}: {msg.content}")



Final State (should only show last 2 messages):

AI: All good here!
HUMAN: Cool, thanks!


In [ ]:
#💥 Why use Pydantic?
# Feature	What it does
# Type safety	Validates that data matches your declared types
# Auto conversion	Converts types when possible ("123" ➝ 123)
# Error messages	Friendly errors if the input is wrong
# Serialization	Easy .json() or .dict() output
# IDE-friendly	Works great with type checkers and autocomplete
# Deep integration	Used in FastAPI, LangChain, LangGraph, SQLModel, etc.

In [77]:
from pydantic import BaseModel

class User(BaseModel):
    name: str
    age: int

# Accepts dict, validates and converts automatically
user = User(name="Alice", age="30")  # str "30" will become int 30

In [79]:
type(user.age)

int